# **City Landscape in Sight: A Crowdsourced Framework for Unlocking Urban-Scale Window View Perceptions from Real Estate Imagery**

**Authors:** [Chucai Peng](https://ual.sg/author/chucai-peng/)†, [Sijie Yang](https://sijie-yang.com)†, Ang Liu, Yang Xiang, Zhixiang Zhou, [Filip Biljecki](https://filipbiljecki.com)*

by [Urban Analytics Lab](https://ual.sg), National University of Singapore

---

(† co-first authors, * corresponding authors)

**Note:** Each .ipynb file can be run independently. [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).


# Code_3_WVI Dataset Sampling (Hexagon & Complex IDs)

Assign spatial sampling IDs to the city-scale WVI table (same H3 logic as **Code 4** hexagon map):

| Column | Description |
|--------|-------------|
| `Hexagon_ID` | Integer 1…*N* from H3 resolution 7 cells; ordered **north→south**, then **west→east** |
| `Complex_ID` | `{Hexagon_ID}-{k}` for each distinct `(Lon, Lat)` within that hexagon (`k` = 1, 2, 3, …; same geographic order) |

**Pipeline position:** run after perception scores are merged into `data/data.csv` (Code 5 inference output → analytics table), before spatial analytics (**Code 4**) or block-CV experiments that need `Complex_ID`.

**Inputs / outputs:**
- **In:** `data/data.csv`
- **Out:** `data/data.csv` (columns inserted after `ID`), `data/data_final_rescaled.csv` (IDs merged on `ID`), `figures/spatial_hexagon_id_map.csv`, `data/metadata/complex_id_map.csv`


In [ ]:
from pathlib import Path

import h3
import pandas as pd

H3_RESOLUTION = 7
DATA_PATH = Path("data/data.csv")
RESCALED_PATH = Path("data/data_final_rescaled.csv")
HEXAGON_MAP_PATH = Path("figures/spatial_hexagon_id_map.csv")
COMPLEX_MAP_PATH = Path("data/metadata/complex_id_map.csv")


In [ ]:
def build_hexagon_id_map(h3_ids):
    """Map H3 cell strings → Simple_ID (1…N), north→south then west→east."""
    unique = list(set(h3_ids))
    centers = {hid: h3.cell_to_latlng(hid) for hid in unique}
    ordered = sorted(unique, key=lambda hid: (-centers[hid][0], centers[hid][1]))
    h3_to_hex = {hid: i + 1 for i, hid in enumerate(ordered)}
    hex_map_df = pd.DataFrame(
        {
            "Simple_ID": range(1, len(ordered) + 1),
            "H3_ID": ordered,
            "Center_Lat": [centers[h][0] for h in ordered],
            "Center_Lon": [centers[h][1] for h in ordered],
        }
    )
    return h3_to_hex, hex_map_df


def assign_hexagon_and_complex_ids(df: pd.DataFrame, h3_resolution: int = H3_RESOLUTION) -> pd.DataFrame:
    """Add Hexagon_ID and Complex_ID; drop prior ID columns if re-running."""
    out = df.drop(columns=["Hexagon_ID", "Complex_ID"], errors="ignore").copy()
    h3_cells = [h3.latlng_to_cell(lat, lon, h3_resolution) for lat, lon in zip(out["Lat"], out["Lon"])]
    h3_to_hex, _ = build_hexagon_id_map(h3_cells)
    out["Hexagon_ID"] = [h3_to_hex[c] for c in h3_cells]

    chunks = []
    for hex_id, group in out.groupby("Hexagon_ID", sort=True):
        locs = (
            group[["Lon", "Lat"]]
            .drop_duplicates()
            .sort_values(by=["Lat", "Lon"], ascending=[False, True])
        )
        loc_to_complex = {
            (row.Lon, row.Lat): f"{hex_id}-{i}"
            for i, (_, row) in enumerate(locs.iterrows(), start=1)
        }
        chunks.append(
            group.assign(
                Complex_ID=[loc_to_complex[(r.Lon, r.Lat)] for _, r in group.iterrows()]
            )
        )
    out = pd.concat(chunks).sort_index()

    front = ["ID", "Hexagon_ID", "Complex_ID"]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows from {DATA_PATH}")

h3_cells = [h3.latlng_to_cell(lat, lon, H3_RESOLUTION) for lat, lon in zip(df["Lat"], df["Lon"])]
_, hex_map_df = build_hexagon_id_map(h3_cells)

df_labeled = assign_hexagon_and_complex_ids(df)
df_labeled.to_csv(DATA_PATH, index=False)
hex_map_df.to_csv(HEXAGON_MAP_PATH, index=False)

complex_map = (
    df_labeled[["Complex_ID", "Hexagon_ID", "Lon", "Lat"]]
    .drop_duplicates(subset=["Complex_ID"])
    .sort_values(["Hexagon_ID", "Complex_ID"])
)
COMPLEX_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
complex_map.to_csv(COMPLEX_MAP_PATH, index=False)

if RESCALED_PATH.exists():
    resc = pd.read_csv(RESCALED_PATH).drop(columns=["Hexagon_ID", "Complex_ID"], errors="ignore")
    resc = df_labeled[["ID", "Hexagon_ID", "Complex_ID"]].merge(resc, on="ID", how="right")
    rrest = [c for c in resc.columns if c not in ("ID", "Hexagon_ID", "Complex_ID")]
    resc[ ["ID", "Hexagon_ID", "Complex_ID"] + rrest ].to_csv(RESCALED_PATH, index=False)
    print(f"Updated {RESCALED_PATH}")

print(f"Saved {DATA_PATH}")
print(f"Hexagon map → {HEXAGON_MAP_PATH} ({len(hex_map_df)} hexagons)")
print(f"Complex lookup → {COMPLEX_MAP_PATH} ({len(complex_map)} unique Lon/Lat locations)")


In [ ]:
n_hex = df_labeled["Hexagon_ID"].nunique()
n_complex = df_labeled["Complex_ID"].nunique()
per_hex = df_labeled.groupby("Hexagon_ID")["Complex_ID"].nunique()

print("Summary")
print("=" * 60)
print(f"  WVI rows:              {len(df_labeled):,}")
print(f"  Unique hexagons:       {n_hex}")
print(f"  Unique complexes:      {n_complex}  (distinct Lon/Lat)")
print(f"  Complexes per hexagon: min={per_hex.min()}, max={per_hex.max()}, median={per_hex.median():.0f}")

example_hex = int(per_hex.idxmax())
print(f"\nExample hexagon {example_hex} (most distinct locations):")
display(
    df_labeled.loc[df_labeled["Hexagon_ID"] == example_hex, ["ID", "Complex_ID", "Lon", "Lat", "Floor"]]
    .drop_duplicates(subset=["Complex_ID"])
    .head(10)
)

df_labeled.head()
